In [ ]:
# pip install --upgrade jupyter ipywidgets


^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 6.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------- ----------- 1.6/2.2 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 5.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ----- ---------------------------------- 1.8/12.4 MB 9.1 MB/s eta 0:00:02
   -------- ------------------------------- 2.6/12.4 MB 6.9 MB/s eta 0:00:02
   ----------- ---------------------------- 3.7/12.4 MB 6.1 MB/s eta 0:00:02
   -------------- ------------------------- 4.5/12.4 MB 5.4 MB/s eta 0:00:02
   ---------------- ----------------------- 5.2/12.4 MB 5.0 MB/s eta 0:00:02
   ------------------- -------------------- 6.0/12.4 MB 4.7 MB/s eta 0:00:02
   ---------------------- ----------------- 6.8/12.4 MB 4.6 MB/s eta 0:00:02
   ----------------

In [ ]:
import re
import time
import logging
from datetime import datetime, timedelta
from pathlib import Path

import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

try:
    from tqdm.notebook import tqdm as notebook_tqdm
except Exception:
    notebook_tqdm = None


def progress_iter(iterable, total: int, desc: str, unit: str):
    if notebook_tqdm is not None:
        return notebook_tqdm(iterable, total=total, desc=desc, unit=unit)
    return iterable


# ---- CONFIG ----
main_link = "https://old.dghs.gov.bd/index.php/bd/home/5200-daily-dengue-status-report"
sub_link_example = "https://old.dghs.gov.bd/images/docs/vpr/20260219_dengue_all.pdf"
end_link_part_1 = "https://old.dghs.gov.bd/images/docs/vpr/20211213_dengue_all.pdf"
end_link_part_2 = "https://old.dghs.gov.bd/images/docs/Notice/2019/dengue/Dengue_27_08_19.pdf"
folder = "PDF_SCRAPPED"

start_date = datetime(2026, 2, 19)
end_date = datetime(2019, 8, 27)
split_date = datetime(2021, 12, 13)  # around where naming/path pattern shifts

# Required gap skip: after 13/12/2021 go directly to 30/01/2020
jump_after_date = datetime(2021, 12, 13).date()
jump_to_date = datetime(2020, 1, 30).date()

output_dir = Path(folder)
output_dir.mkdir(parents=True, exist_ok=True)

log_file = output_dir / f"download_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(log_file, encoding="utf-8"),
    ],
)
logger = logging.getLogger("dengue_pdf_scraper")

logger.info("Main page reference: %s", main_link)
logger.info("Part-1 sample: %s", sub_link_example)
logger.info("Download folder: %s", output_dir.resolve())
logger.info("Configured date range: %s to %s", start_date.date(), end_date.date())
logger.info("Gap rule: after %s jump to %s", jump_after_date, jump_to_date)

canonical_pattern = re.compile(r"^(\d{8})_dengue_all\.pdf$", re.IGNORECASE)
legacy_pattern_ddmmyy = re.compile(r"^Dengue_(\d{2})_(\d{2})_(\d{2})\.pdf$", re.IGNORECASE)
legacy_pattern_yyyymmdd = re.compile(r"^Dengue_(\d{8})\.pdf$", re.IGNORECASE)
legacy_pattern_yyyy_mm_dd = re.compile(r"^Dengue_(\d{4})_(\d{2})_(\d{2})\.pdf$", re.IGNORECASE)


def canonical_filename(date_obj: datetime) -> str:
    return f"{date_obj.strftime('%Y%m%d')}_dengue_all.pdf"


def part1_url(date_obj: datetime) -> str:
    return f"https://old.dghs.gov.bd/images/docs/vpr/{date_obj.strftime('%Y%m%d')}_dengue_all.pdf"


def part2_url_ddmmyy(date_obj: datetime) -> str:
    return f"https://old.dghs.gov.bd/images/docs/Notice/2019/dengue/Dengue_{date_obj.strftime('%d_%m_%y')}.pdf"


def part2_url_yyyymmdd(date_obj: datetime) -> str:
    return f"https://old.dghs.gov.bd/images/docs/Notice/2019/dengue/Dengue_{date_obj.strftime('%Y%m%d')}.pdf"


def part2_url_yyyy_mm_dd(date_obj: datetime) -> str:
    return f"https://old.dghs.gov.bd/images/docs/Notice/2019/dengue/Dengue_{date_obj.strftime('%Y_%m_%d')}.pdf"


def candidate_urls(date_obj: datetime):
    p1 = part1_url(date_obj)
    p2_ddmmyy = part2_url_ddmmyy(date_obj)
    p2_yyyymmdd = part2_url_yyyymmdd(date_obj)
    p2_yyyy_mm_dd = part2_url_yyyy_mm_dd(date_obj)

    # Prefer vpr after split date, else Notice patterns first
    if date_obj >= split_date:
        urls = [p1, p2_ddmmyy, p2_yyyymmdd, p2_yyyy_mm_dd]
    else:
        urls = [p2_ddmmyy, p2_yyyymmdd, p2_yyyy_mm_dd, p1]

    # De-duplicate while preserving order
    return list(dict.fromkeys(urls))


def setup_driver(download_path: Path):
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--window-size=1920,1080")

    prefs = {
        "download.default_directory": str(download_path.resolve()),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "plugins.always_open_pdf_externally": True,
        "safebrowsing.enabled": True,
    }
    chrome_options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(30)
    return driver


def list_files(folder_path: Path):
    return {p.name for p in folder_path.iterdir() if p.is_file()}


def wait_for_stable_download(folder_path: Path, before_files: set, timeout: int = 15):
    end_time = time.time() + timeout
    last_seen_new_pdf = None

    while time.time() < end_time:
        current_files = list(folder_path.iterdir())

        # If any temporary Chrome download exists, keep waiting
        if any(p.suffix == ".crdownload" for p in current_files):
            time.sleep(0.5)
            continue

        # Find newly created/finished PDF files
        new_pdf_files = [
            p for p in current_files
            if p.is_file() and p.suffix.lower() == ".pdf" and p.name not in before_files
        ]

        if new_pdf_files:
            # take most recently modified file
            last_seen_new_pdf = max(new_pdf_files, key=lambda x: x.stat().st_mtime)
            return last_seen_new_pdf

        time.sleep(0.5)

    return last_seen_new_pdf


def looks_like_pdf(content: bytes) -> bool:
    return content.startswith(b"%PDF")


def download_with_requests(url: str, target_path: Path, timeout: int = 30):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
    }

    response = requests.get(url, timeout=timeout, headers=headers)
    if response.status_code != 200:
        return False, f"HTTP {response.status_code}"

    content_type = response.headers.get("Content-Type", "").lower()
    content = response.content

    if "pdf" in content_type or looks_like_pdf(content):
        target_path.write_bytes(content)
        return True, "saved"

    return False, f"Not a PDF (Content-Type={content_type})"


def download_with_selenium(driver, url: str, target_path: Path):
    before_files = list_files(output_dir)

    try:
        driver.get(url)
        downloaded_file = wait_for_stable_download(output_dir, before_files, timeout=18)

        if downloaded_file and downloaded_file.exists():
            if downloaded_file.resolve() != target_path.resolve():
                if target_path.exists():
                    target_path.unlink()
                downloaded_file.rename(target_path)
            return True, "selenium_downloaded"

        return False, "selenium_no_pdf_file"

    except Exception as ex:
        return False, f"selenium_error: {ex}"


def in_forced_gap(day_value):
    return jump_to_date < day_value < jump_after_date


def apply_gap_skip(day_value):
    if in_forced_gap(day_value):
        return jump_to_date
    return day_value


def daterange_desc(start_dt: datetime, end_dt: datetime):
    current_date = apply_gap_skip(start_dt.date())

    while current_date >= end_dt.date():
        yield datetime.combine(current_date, datetime.min.time())

        if current_date == jump_after_date:
            current_date = jump_to_date
        else:
            current_date = current_date - timedelta(days=1)
            current_date = apply_gap_skip(current_date)


def count_effective_days(start_dt: datetime, end_dt: datetime):
    return sum(1 for _ in daterange_desc(start_dt, end_dt))


def extract_date_from_filename(file_name: str):
    m1 = canonical_pattern.match(file_name)
    if m1:
        try:
            return datetime.strptime(m1.group(1), "%Y%m%d").date()
        except ValueError:
            return None

    m2 = legacy_pattern_ddmmyy.match(file_name)
    if m2:
        day, month, year_2d = m2.groups()
        try:
            return datetime.strptime(f"{day}_{month}_{year_2d}", "%d_%m_%y").date()
        except ValueError:
            return None

    m3 = legacy_pattern_yyyymmdd.match(file_name)
    if m3:
        try:
            return datetime.strptime(m3.group(1), "%Y%m%d").date()
        except ValueError:
            return None

    m4 = legacy_pattern_yyyy_mm_dd.match(file_name)
    if m4:
        year, month, day = m4.groups()
        try:
            return datetime.strptime(f"{year}_{month}_{day}", "%Y_%m_%d").date()
        except ValueError:
            return None

    return None


def build_existing_index(folder_path: Path):
    existing = {}
    for pdf_file in folder_path.glob("*.pdf"):
        detected_date = extract_date_from_filename(pdf_file.name)
        if detected_date is None:
            continue
        existing.setdefault(detected_date, []).append(pdf_file)
    return existing


def normalize_existing_files_for_date(day_dt: datetime, files_for_date):
    target = output_dir / canonical_filename(day_dt)
    if target.exists():
        return True

    if len(files_for_date) == 1:
        src = files_for_date[0]
        if src.exists() and src.resolve() != target.resolve():
            src.rename(target)
            logger.info("[%s] Renamed existing file to canonical: %s", day_dt.date(), target.name)
        return target.exists()

    return False


def compute_resume_start(existing_map, configured_start: datetime, configured_end: datetime):
    if not existing_map:
        logger.info("No existing PDF dates found. Starting from configured start date: %s", configured_start.date())
        return configured_start

    lowest_existing_date = min(existing_map.keys())
    resume_date = lowest_existing_date - timedelta(days=1)
    logger.info("Lowest existing downloaded date found: %s", lowest_existing_date)

    if resume_date < configured_end.date():
        logger.info("All remaining dates are below configured end date. Nothing left to download.")
        return None

    if resume_date > configured_start.date():
        resume_date = configured_start.date()

    adjusted_resume_date = apply_gap_skip(resume_date)
    if adjusted_resume_date != resume_date:
        logger.info("Resume date %s is in forced gap. Adjusted to %s", resume_date, adjusted_resume_date)
        resume_date = adjusted_resume_date

    logger.info("Resume start date selected: %s", resume_date)
    return datetime.combine(resume_date, datetime.min.time())


configured_total_days = count_effective_days(start_date, end_date)
logger.info("Configured total dates (after gap rule): %d", configured_total_days)

existing_by_date = build_existing_index(output_dir)
logger.info("Pre-scan complete: %d existing date(s) detected in download folder.", len(existing_by_date))

run_start_date = compute_resume_start(existing_by_date, start_date, end_date)

stats = {
    "total_dates": 0,
    "already_exists": 0,
    "downloaded_selenium": 0,
    "downloaded_requests": 0,
    "missing": 0,
    "errors": 0,
}

missing_dates = []
error_dates = []

if run_start_date is None:
    logger.info("Skipping download loop because continuation point is below configured end date.")
else:
    run_total_days = count_effective_days(run_start_date, end_date)
    logger.info("Dates to process this run: %d (from %s down to %s)", run_total_days, run_start_date.date(), end_date.date())

    driver = setup_driver(output_dir)
    logger.info("Headless Selenium driver initialized.")

    try:
        for day in progress_iter(
            daterange_desc(run_start_date, end_date),
            total=run_total_days,
            desc="Downloading dengue PDFs",
            unit="day",
        ):
            stats["total_dates"] += 1
            target_name = canonical_filename(day)
            target_path = output_dir / target_name
            day_key = day.date()

            existing_files = existing_by_date.get(day_key, [])
            if existing_files:
                normalized = normalize_existing_files_for_date(day, existing_files)
                stats["already_exists"] += 1
                if normalized:
                    logger.info("[%s] already downloaded -> %s", day.date(), target_name)
                else:
                    logger.info("[%s] already downloaded (multiple/legacy names kept) -> skip", day.date())
                continue

            urls = candidate_urls(day)
            logger.info("\n[%s] Trying URLs in order:", day.date())
            for idx, u in enumerate(urls, start=1):
                logger.info("  %d) %s", idx, u)

            day_done = False
            day_error = False

            for url in urls:
                # 1) Selenium primary attempt
                ok_sel, msg_sel = download_with_selenium(driver, url, target_path)
                logger.info("[%s] Selenium -> %s | %s", day.date(), "OK" if ok_sel else "FAIL", msg_sel)

                if ok_sel and target_path.exists():
                    stats["downloaded_selenium"] += 1
                    existing_by_date.setdefault(day_key, []).append(target_path)
                    logger.info("[%s] Saved (normalized): %s", day.date(), target_name)
                    day_done = True
                    break

                # 2) Requests fallback
                ok_req, msg_req = download_with_requests(url, target_path)
                logger.info("[%s] Requests fallback -> %s | %s", day.date(), "OK" if ok_req else "FAIL", msg_req)

                if ok_req and target_path.exists():
                    stats["downloaded_requests"] += 1
                    existing_by_date.setdefault(day_key, []).append(target_path)
                    logger.info("[%s] Saved via fallback (normalized): %s", day.date(), target_name)
                    day_done = True
                    break

                # If URL failed by both methods, move to next candidate URL
                if "selenium_error" in msg_sel:
                    day_error = True

            if not day_done:
                if day_error:
                    stats["errors"] += 1
                    error_dates.append(day.strftime("%Y-%m-%d"))
                    logger.error("[%s] Failed with at least one Selenium error and no valid PDF downloaded.", day.date())
                else:
                    stats["missing"] += 1
                    missing_dates.append(day.strftime("%Y-%m-%d"))
                    logger.warning("[%s] No PDF found for this date (likely missing report).", day.date())

    finally:
        driver.quit()
        logger.info("Selenium driver closed.")

logger.info("\n===== FINAL SUMMARY =====")
logger.info("Total dates checked      : %d", stats["total_dates"])
logger.info("Already existed          : %d", stats["already_exists"])
logger.info("Downloaded via Selenium  : %d", stats["downloaded_selenium"])
logger.info("Downloaded via Requests  : %d", stats["downloaded_requests"])
logger.info("Missing dates            : %d", stats["missing"])
logger.info("Error dates              : %d", stats["errors"])

if missing_dates:
    logger.info("\nMissing date list:")
    for d in missing_dates:
        logger.info("  - %s", d)

if error_dates:
    logger.info("\nError date list:")
    for d in error_dates:
        logger.info("  - %s", d)

logger.info("Log file saved at: %s", log_file.resolve())

2026-02-21 06:09:56,581 | INFO | Main page reference: https://old.dghs.gov.bd/index.php/bd/home/5200-daily-dengue-status-report
2026-02-21 06:09:56,582 | INFO | Part-1 sample: https://old.dghs.gov.bd/images/docs/vpr/20260219_dengue_all.pdf
2026-02-21 06:09:56,585 | INFO | Download folder: D:\ALL CODES\MATLAB PROJECT\PDF_SCRAPPED
2026-02-21 06:09:56,585 | INFO | Configured date range: 2026-02-19 to 2019-08-27
2026-02-21 06:09:56,587 | INFO | Gap rule: after 2021-12-13 jump to 2021-02-04
2026-02-21 06:09:56,593 | INFO | Configured total dates (after gap rule): 2058
2026-02-21 06:09:56,612 | INFO | Pre-scan complete: 1506 existing date(s) detected in download folder.
2026-02-21 06:09:56,614 | INFO | Lowest existing downloaded date found: 2021-12-13
2026-02-21 06:09:56,614 | INFO | Resume date 2021-12-12 is in forced gap. Adjusted to 2021-02-04
2026-02-21 06:09:56,615 | INFO | Resume start date selected: 2021-02-04
2026-02-21 06:09:56,617 | INFO | Dates to process this run: 528 (from 2021-

2026-02-21 06:09:57,924 | INFO | 
[2021-02-04] Trying URLs in order:
2026-02-21 06:09:57,925 | INFO |   1) https://old.dghs.gov.bd/images/docs/Notice/2019/dengue/Dengue_04_02_21.pdf
2026-02-21 06:09:57,925 | INFO |   2) https://old.dghs.gov.bd/images/docs/Notice/2019/dengue/Dengue_20210204.pdf
2026-02-21 06:09:57,926 | INFO |   3) https://old.dghs.gov.bd/images/docs/Notice/2019/dengue/Dengue_2021_02_04.pdf
2026-02-21 06:09:57,926 | INFO |   4) https://old.dghs.gov.bd/images/docs/vpr/20210204_dengue_all.pdf
2026-02-21 06:10:22,050 | INFO | [2021-02-04] Selenium -> FAIL | selenium_no_pdf_file
2026-02-21 06:10:22,095 | INFO | [2021-02-04] Requests fallback -> FAIL | HTTP 404
2026-02-21 06:10:40,572 | INFO | [2021-02-04] Selenium -> FAIL | selenium_no_pdf_file
2026-02-21 06:10:40,626 | INFO | [2021-02-04] Requests fallback -> FAIL | HTTP 404
2026-02-21 06:10:41,279 | INFO | [2021-02-04] Selenium -> OK | selenium_downloaded
2026-02-21 06:10:41,281 | INFO | [2021-02-04] Saved (normalized): 2

KeyboardInterrupt: 

In [ ]:
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path
import re

# ---- VALIDATION CONFIG ----
validation_folder = Path("PDF_SCRAPPED")
expected_start = datetime(2026, 2, 19)
expected_end = datetime(2019, 8, 27)
filename_pattern = re.compile(r"^(\d{8})_dengue_all\.pdf$", re.IGNORECASE)


def expected_dates_desc(start_dt: datetime, end_dt: datetime):
    cur = start_dt
    while cur >= end_dt:
        yield cur.date()
        cur -= timedelta(days=1)


def is_pdf_header_ok(file_path: Path) -> bool:
    try:
        with file_path.open("rb") as f:
            return f.read(5) == b"%PDF-"
    except Exception:
        return False


all_pdfs = sorted(validation_folder.glob("*.pdf"))
matched_files = []
unexpected_name_files = []
corrupt_files = []

date_to_files = defaultdict(list)

for pdf_file in all_pdfs:
    name = pdf_file.name
    m = filename_pattern.match(name)

    if not m:
        unexpected_name_files.append(name)
        continue

    date_str = m.group(1)
    try:
        date_obj = datetime.strptime(date_str, "%Y%m%d").date()
    except ValueError:
        unexpected_name_files.append(name)
        continue

    matched_files.append(name)
    date_to_files[date_obj].append(name)

    if not is_pdf_header_ok(pdf_file):
        corrupt_files.append(name)

expected_set = set(expected_dates_desc(expected_start, expected_end))
downloaded_set = set(date_to_files.keys())

missing_dates = sorted(expected_set - downloaded_set)
extra_dates = sorted(downloaded_set - expected_set)
duplicate_dates = {d: files for d, files in date_to_files.items() if len(files) > 1}

missing_by_month = defaultdict(int)
for d in missing_dates:
    missing_by_month[d.strftime("%Y-%m")] += 1

print("===== PDF DOWNLOAD VALIDATION REPORT =====")
print(f"Validation folder            : {validation_folder.resolve()}")
print(f"Expected date range          : {expected_end} to {expected_start}")
print(f"Expected total days          : {len(expected_set)}")
print(f"Total .pdf files found       : {len(all_pdfs)}")
print(f"Canonical-name PDFs          : {len(matched_files)}")
print(f"Unexpected-name PDFs         : {len(unexpected_name_files)}")
print(f"Unique dates downloaded      : {len(downloaded_set)}")
print(f"Missing dates                : {len(missing_dates)}")
print(f"Extra dates (out of range)   : {len(extra_dates)}")
print(f"Duplicate-date groups        : {len(duplicate_dates)}")
print(f"Likely corrupt PDFs (header) : {len(corrupt_files)}")

if missing_by_month:
    print("\n--- Missing dates by month ---")
    for month_key in sorted(missing_by_month.keys()):
        print(f"{month_key}: {missing_by_month[month_key]}")

if missing_dates:
    print("\n--- First 30 missing dates ---")
    for d in missing_dates[:30]:
        print(d)

if duplicate_dates:
    print("\n--- Duplicate date files ---")
    for d, files in sorted(duplicate_dates.items()):
        print(f"{d}: {files}")

if unexpected_name_files:
    print("\n--- Unexpected filename pattern (first 30) ---")
    for name in unexpected_name_files[:30]:
        print(name)

if extra_dates:
    print("\n--- Out-of-range dates (first 30) ---")
    for d in extra_dates[:30]:
        print(d)

if corrupt_files:
    print("\n--- Likely corrupt PDFs (first 30) ---")
    for name in corrupt_files[:30]:
        print(name)

print("\nValidation complete.")